In [1]:
import torch
import torch.nn as nn
import math

In [2]:
import torch
import torch.nn as nn
import math

# ==============================
# Positional Encoding
# ==============================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [3]:
# ==============================
# Scaled Dot-Product Attention
# ==============================
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attention = torch.softmax(scores, dim=-1)
    output = torch.matmul(attention, V)
    return output, attention


In [4]:
# ==============================
# Multi-Head Attention
# ==============================
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch_size, seq_len, d_model = x.size()
        return x.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, num_heads, seq_len, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        attn_output, _ = scaled_dot_product_attention(Q, K, V, mask)
        output = self.W_o(self.combine_heads(attn_output))
        return output


In [5]:
# ==============================
# Feed Forward Network
# ==============================
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(FeedForward, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)



In [6]:
# ==============================
# Encoder Layer
# ==============================
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn))

        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x

In [7]:
# ==============================
# Decoder Layer
# ==============================
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        attn1 = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn1))

        attn2 = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn2))

        ff_out = self.ff(x)
        x = self.norm3(x + self.dropout(ff_out))
        return x

In [8]:
# ==============================
# Encoder
# ==============================
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers,
                 num_heads, d_ff, dropout, max_len):
        super(Encoder, self).__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = self.embedding(x) * math.sqrt(self.embedding.embedding_dim)
        x = self.pos_encoding(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, mask)

        return x


In [9]:
# ==============================
# Decoder
# ==============================
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers,
                 num_heads, d_ff, dropout, max_len):
        super(Decoder, self).__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)

        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        x = self.embedding(x) * math.sqrt(self.embedding.embedding_dim)
        x = self.pos_encoding(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)

        return x


In [10]:
# ==============================
# Transformer Model
# ==============================
class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        num_layers=6,
        num_heads=8,
        d_ff=2048,
        dropout=0.1,
        max_len=5000,
    ):
        super(Transformer, self).__init__()

        self.encoder = Encoder(
            src_vocab_size, d_model, num_layers,
            num_heads, d_ff, dropout, max_len
        )

        self.decoder = Decoder(
            tgt_vocab_size, d_model, num_layers,
            num_heads, d_ff, dropout, max_len
        )

        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def generate_src_mask(self, src):
        return (src != 0).unsqueeze(1).unsqueeze(2)

    def generate_tgt_mask(self, tgt):
        seq_len = tgt.size(1)
        tgt_mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
        return tgt_mask.unsqueeze(0).unsqueeze(1)

    def forward(self, src, tgt):
        src_mask = self.generate_src_mask(src).to(src.device)
        tgt_mask = self.generate_tgt_mask(tgt).to(tgt.device)

        enc_output = self.encoder(src, src_mask)
        dec_output = self.decoder(tgt, enc_output, src_mask, tgt_mask)

        output = self.fc_out(dec_output)
        return output

In [11]:
# ==============================
# Testing the Model
# ==============================
if __name__ == "__main__":
    src_vocab_size = 10000
    tgt_vocab_size = 10000
    src = torch.randint(0, src_vocab_size, (2, 10))
    tgt = torch.randint(0, tgt_vocab_size, (2, 10))

    model = Transformer(src_vocab_size, tgt_vocab_size)
    out = model(src, tgt)

    print("Output shape:", out.shape)

Output shape: torch.Size([2, 10, 10000])


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import math
import numpy as np



def test_model_basics():
    """Test 1: Basic forward pass and shape checking"""
    print("Test 1: Basic Forward Pass")
    src_vocab_size = 10000
    tgt_vocab_size = 10000
    
    # Test different batch sizes and sequence lengths
    test_cases = [
        (2, 10),   # Original test
        (4, 20),   # Larger batch/seq
        (1, 5),    # Small batch/seq
        (8, 50),   # Large batch/seq
    ]
    
    model = Transformer(src_vocab_size, tgt_vocab_size)
    model.eval()
    
    for batch_size, seq_len in test_cases:
        src = torch.randint(0, src_vocab_size, (batch_size, seq_len))
        tgt = torch.randint(0, tgt_vocab_size, (batch_size, seq_len))
        
        with torch.no_grad():
            out = model(src, tgt)
        
        expected_shape = (batch_size, seq_len, tgt_vocab_size)
        assert out.shape == expected_shape, f"Expected {expected_shape}, got {out.shape}"
        print(f"  ✓ Batch {batch_size}, Seq {seq_len} -> {out.shape}")
    
    print("Test 1 PASSED\n")


In [13]:
def test_padding_mask():
    """Test 2: Padding mask functionality"""
    print("Test 2: Padding Mask")
    model = Transformer(10000, 10000)
    model.eval()
    
    # Create input with padding (0 tokens)
    batch_size, seq_len = 3, 8
    src = torch.tensor([
        [1, 2, 3, 0, 0, 0, 0, 0],  # 3 real tokens
        [4, 5, 6, 7, 0, 0, 0, 0],  # 4 real tokens
        [8, 9, 0, 0, 0, 0, 0, 0],  # 2 real tokens
    ])
    tgt = torch.tensor([
        [10, 11, 12, 13, 14, 0, 0, 0],
        [15, 16, 17, 0, 0, 0, 0, 0],
        [18, 19, 20, 21, 22, 23, 0, 0],
    ])
    
    with torch.no_grad():
        out = model(src, tgt)
    
    print(f"  ✓ Padding mask works: Output shape {out.shape}")
    print("Test 2 PASSED\n")


In [14]:
def test_training_step():
    """Test 3: Training step with loss computation"""
    print(" Test 3: Training Step")
    model = Transformer(5000, 5000, d_model=128, num_layers=2)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding
    
    # Simple synthetic data
    batch_size, seq_len = 4, 12
    src = torch.randint(1, 1000, (batch_size, seq_len))
    tgt_input = torch.randint(1, 1000, (batch_size, seq_len))
    tgt_output = torch.randint(1, 1000, (batch_size, seq_len))  # Shifted target
    
    model.train()
    optimizer.zero_grad()
    
    output = model(src, tgt_input)
    output = output.reshape(-1, output.size(-1))  # [batch*seq, vocab]
    target = tgt_output.reshape(-1)  # [batch*seq]
    
    loss = criterion(output, target)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    print(f"  ✓ Training loss: {loss.item():.4f}")
    print(f"  ✓ Grad norm: {torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1000):.4f}")
    print(" Test 3 PASSED\n")


In [15]:
def test_memory_efficiency():
    """Test 4: Memory usage with large sequences"""
    print(" Test 4: Memory Efficiency")
    model = Transformer(10000, 10000, d_model=256, num_layers=3)
    
    batch_size, seq_len = 2, 200  # Large sequence
    src = torch.randint(0, 10000, (batch_size, seq_len))
    tgt = torch.randint(0, 10000, (batch_size, seq_len))
    
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    model.eval()
    with torch.no_grad():
        out = model(src, tgt)
    
    print(f"  ✓ Large seq (200): Output shape {out.shape}")
    print("Test 4 PASSED\n")


In [16]:

if __name__ == "__main__":
    print("🚀 Testing Transformer Implementation\n")
    
    # Run all tests
    test_model_basics()
    test_padding_mask()
    test_training_step()
    test_memory_efficiency()
    
    print("🎉 ALL TESTS PASSED! Your Transformer is ready to use!")

🚀 Testing Transformer Implementation

Test 1: Basic Forward Pass
  ✓ Batch 2, Seq 10 -> torch.Size([2, 10, 10000])
  ✓ Batch 4, Seq 20 -> torch.Size([4, 20, 10000])
  ✓ Batch 1, Seq 5 -> torch.Size([1, 5, 10000])
  ✓ Batch 8, Seq 50 -> torch.Size([8, 50, 10000])
Test 1 PASSED

Test 2: Padding Mask
  ✓ Padding mask works: Output shape torch.Size([3, 8, 10000])
Test 2 PASSED

 Test 3: Training Step
  ✓ Training loss: 8.5425
  ✓ Grad norm: 1.0000
 Test 3 PASSED

 Test 4: Memory Efficiency
  ✓ Large seq (200): Output shape torch.Size([2, 200, 10000])
Test 4 PASSED

🎉 ALL TESTS PASSED! Your Transformer is ready to use!
